In [ ]:
# Cell 1: Install libraries
# (rdkit provides local cheminformatics; requests is used for the PubChem API call)
%pip install -q rdkit requests


In [40]:
# Cell 2: Imports
import csv
import io
import re
import sys
import time

import requests
from rdkit import Chem, RDLogger, rdBase
from rdkit.Chem import rdMolDescriptors, Descriptors

# NOTE: call order matters - DisableLog must run BEFORE LogToPythonStderr,
# otherwise the disable wins and nothing gets captured.
RDLogger.DisableLog("rdApp.*")
rdBase.LogToPythonStderr()

_TIMESTAMP_RE = re.compile(r"^\[\d{2}:\d{2}:\d{2}\]\s*")

def _clean_rdkit_log(raw: str) -> str:
    if not raw:
        return ""
    lines = [_TIMESTAMP_RE.sub("", ln).strip() for ln in raw.splitlines()]
    lines = [ln for ln in lines if ln]
    deduped = []
    for ln in lines:
        if not deduped or deduped[-1] != ln:
            deduped.append(ln)
    return " | ".join(deduped)[:400]

INPUT_PATH = "/content/input.csv"
OUTPUT_PATH = "output.csv"
OUTPUT_COLUMNS = [
    "compound_id", "input_smiles", "canonical_smiles", "iupac_name",
    "molecular_formula", "exact_mass", "inchikey", "status", "notes",
]

In [42]:
# Cell 3: Load input
with open(INPUT_PATH, newline="") as f:
    input_rows = list(csv.DictReader(f))

print(f"Loaded {len(input_rows)} rows from {INPUT_PATH}")
input_rows


Loaded 15 rows from /content/input.csv


[{'compound_id': 'CMP-001', 'smiles': 'CCO'},
 {'compound_id': 'CMP-002', 'smiles': 'CC(C)O'},
 {'compound_id': 'CMP-003', 'smiles': 'CC(=O)Oc1ccccc1C(=O)O'},
 {'compound_id': 'CMP-004', 'smiles': 'O=C(O)c1ccccc1O'},
 {'compound_id': 'CMP-005', 'smiles': 'CCOC(=O)c1ccc(N)cc1'},
 {'compound_id': 'CMP-006', 'smiles': 'CN(C)C(=O)c1ccccc1'},
 {'compound_id': 'CMP-007', 'smiles': 'O=[N+]([O-])c1ccc(Cl)cc1'},
 {'compound_id': 'CMP-008', 'smiles': 'CC(C)(C)OC(=O)N1CCCCC1'},
 {'compound_id': 'CMP-009', 'smiles': 'CN1CCN(CC1)c1ccccc1'},
 {'compound_id': 'CMP-010', 'smiles': 'CC(C)Cc1ccc(cc1)C(C)C(=O)O'},
 {'compound_id': 'CMP-011', 'smiles': 'C[C@H](O)C(=O)O'},
 {'compound_id': 'CMP-012', 'smiles': 'C1CCNC1'},
 {'compound_id': 'CMP-013', 'smiles': '[Na+].[O-]C(=O)c1ccccc1'},
 {'compound_id': 'CMP-014', 'smiles': 'C1=CC=CC=C1C('},
 {'compound_id': 'CMP-015', 'smiles': 'CCO'}]

In [43]:
# Cell 4: SMILES validation + local properties (RDKit only, no network)

def _parse_with_diagnostics(smiles: str):
    buf = io.StringIO()
    old_stderr = sys.stderr
    sys.stderr = buf
    try:
        mol = Chem.MolFromSmiles(smiles, sanitize=False)
    finally:
        sys.stderr = old_stderr

    if mol is None:
        log = _clean_rdkit_log(buf.getvalue())
        return None, log or "RDKit failed to parse this SMILES (no further detail available)."

    buf2 = io.StringIO()
    sys.stderr = buf2
    try:
        Chem.SanitizeMol(mol)
        sys.stderr = old_stderr
        return mol, None
    except Exception as exc:
        sys.stderr = old_stderr
        detail = str(exc).strip() or _clean_rdkit_log(buf2.getvalue())
        return None, detail or "RDKit sanitization failed (no further detail available)."


def validate_and_compute(smiles: str) -> dict:
    if smiles is None or not smiles.strip():
        return {"valid": False, "diagnostic": "Empty SMILES string."}

    mol, diagnostic = _parse_with_diagnostics(smiles)
    if mol is None:
        return {"valid": False, "diagnostic": diagnostic}

    if mol.GetNumAtoms() == 0:
        return {"valid": False, "diagnostic": "Parsed to an empty molecule (no atoms)."}

    canonical_smiles = Chem.MolToSmiles(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    exact_mass = round(Descriptors.ExactMolWt(mol), 4)

    try:
        inchikey = Chem.MolToInchiKey(mol)
    except Exception:
        inchikey = ""

    n_fragments = len(Chem.GetMolFrags(mol, asMols=False))

    return {
        "valid": True,
        "canonical_smiles": canonical_smiles,
        "molecular_formula": formula,
        "exact_mass": exact_mass,
        "inchikey": inchikey,
        "is_multi_component": n_fragments > 1,
    }


print(validate_and_compute("CCO"))
print(validate_and_compute("C1=CC=CC=C1C("))
print(validate_and_compute(""))
print(validate_and_compute("C(C)(C)(C)(C)C"))

{'valid': True, 'canonical_smiles': 'CCO', 'molecular_formula': 'C2H6O', 'exact_mass': 46.0419, 'inchikey': 'LFQSCWFLJHTTHZ-UHFFFAOYSA-N', 'is_multi_component': False}
{'valid': False, 'diagnostic': "SMILES Parse Error: syntax error while parsing: C1=CC=CC=C1C( | SMILES Parse Error: check for mistakes around position 13: | C1=CC=CC=C1C( | ~~~~~~~~~~~~^ | SMILES Parse Error: Failed parsing SMILES 'C1=CC=CC=C1C(' for input: 'C1=CC=CC=C1C('"}
{'valid': False, 'diagnostic': 'Empty SMILES string.'}
{'valid': False, 'diagnostic': 'Explicit valence for atom # 0 C, 5, is greater than permitted'}


In [44]:
# Cell 5: PubChem IUPAC name lookup (external, cached, failure-tolerant)

PUBCHEM_URL = (
    "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/"
    "inchikey/{inchikey}/property/IUPACName/JSON"
)

def lookup_iupac_name(inchikey: str, cache: dict, session: requests.Session,
                       max_retries: int = 2, delay: float = 0.25):
    if not inchikey:
        return None, "ERROR", "No InChIKey available; IUPAC lookup skipped."

    if inchikey in cache:
        return cache[inchikey]

    url = PUBCHEM_URL.format(inchikey=inchikey)
    name, lookup_status, note = None, "ERROR", "PubChem: lookup did not complete."

    for attempt in range(max_retries + 1):
        try:
            resp = session.get(url, timeout=15)
            if resp.status_code == 200:
                data = resp.json()
                name = data["PropertyTable"]["Properties"][0]["IUPACName"]
                lookup_status = "FOUND"
                note = "PubChem: IUPAC name resolved via InChIKey."
            elif resp.status_code == 404:
                lookup_status = "NOT_FOUND"
                note = "PubChem: no matching compound for this InChIKey."
            else:
                lookup_status = "ERROR"
                note = f"PubChem: request failed (HTTP {resp.status_code})."
            break
        except requests.RequestException as exc:
            lookup_status = "ERROR"
            note = f"PubChem: request error ({type(exc).__name__}); giving up after retries."
            if attempt < max_retries:
                time.sleep(delay)
                continue
        time.sleep(delay)

    result = (name, lookup_status, note)
    cache[inchikey] = result
    return result

In [45]:
# Cell 6: Main processing loop

name_cache: dict = {}
session = requests.Session()
output_rows = []

for row in input_rows:
    compound_id = row["compound_id"]
    input_smiles = row["smiles"]

    local = validate_and_compute(input_smiles)

    if not local["valid"]:
        output_rows.append({
            "compound_id": compound_id,
            "input_smiles": input_smiles,
            "canonical_smiles": "",
            "iupac_name": "",
            "molecular_formula": "",
            "exact_mass": "",
            "inchikey": "",
            "status": "INVALID_SMILES",
            "notes": f"RDKit could not process this SMILES: {local['diagnostic']}",
        })
        continue

    inchikey = local["inchikey"]
    was_cached = inchikey in name_cache
    name, lookup_status, lookup_note = lookup_iupac_name(inchikey, name_cache, session)

    if local["is_multi_component"]:
        status = "MULTI_COMPONENT"
    elif lookup_status == "FOUND":
        status = "SUCCESS"
    elif lookup_status == "NOT_FOUND":
        status = "NOT_FOUND"
    else:
        status = "LOOKUP_ERROR"

    note_parts = []
    if local["is_multi_component"]:
        note_parts.append("Multiple components detected (e.g. salt/mixture); full structure preserved, not split.")
        if lookup_status == "FOUND":
            note_parts.append(
                "External lookup returned a name for this InChIKey; for a multi-component/"
                "disconnected structure this may not represent the entire combined structure "
                "— review recommended, no name was invented for the full structure."
            )
        elif lookup_status == "NOT_FOUND":
            note_parts.append("External lookup found no matching name for this InChIKey.")
        else:
            note_parts.append(lookup_note)
    if was_cached:
        note_parts.append("Duplicate structure; reused cached PubChem result, no repeated API call.")
    if not local["is_multi_component"]:
        note_parts.append(lookup_note)

    output_rows.append({
        "compound_id": compound_id,
        "input_smiles": input_smiles,
        "canonical_smiles": local["canonical_smiles"],
        "iupac_name": name or "",
        "molecular_formula": local["molecular_formula"],
        "exact_mass": local["exact_mass"],
        "inchikey": inchikey,
        "status": status,
        "notes": " ".join(p for p in note_parts if p),
    })

print(f"Processed {len(output_rows)} rows")

Processed 15 rows


In [46]:
# Cell 7: Generate output.csv + validation summary
from collections import Counter

with open(OUTPUT_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=OUTPUT_COLUMNS)
    writer.writeheader()
    writer.writerows(output_rows)

print(f"Wrote {len(output_rows)} rows to {OUTPUT_PATH}")

status_counts = Counter(r["status"] for r in output_rows)
print("Status counts:", dict(status_counts))

assert len(output_rows) == len(input_rows), "row count mismatch vs input.csv"
with open(OUTPUT_PATH, newline="") as f:
    header = next(csv.reader(f))
assert header == OUTPUT_COLUMNS, "output.csv column mismatch"
print("OK: row count and column order match the spec.")
output_rows

Wrote 15 rows to output.csv
Status counts: {'SUCCESS': 13, 'MULTI_COMPONENT': 1, 'INVALID_SMILES': 1}
OK: row count and column order match the spec.


[{'compound_id': 'CMP-001',
  'input_smiles': 'CCO',
  'canonical_smiles': 'CCO',
  'iupac_name': 'ethanol',
  'molecular_formula': 'C2H6O',
  'exact_mass': 46.0419,
  'inchikey': 'LFQSCWFLJHTTHZ-UHFFFAOYSA-N',
  'status': 'SUCCESS',
  'notes': 'PubChem: IUPAC name resolved via InChIKey.'},
 {'compound_id': 'CMP-002',
  'input_smiles': 'CC(C)O',
  'canonical_smiles': 'CC(C)O',
  'iupac_name': 'propan-2-ol',
  'molecular_formula': 'C3H8O',
  'exact_mass': 60.0575,
  'inchikey': 'KFZMGEQAYNKOFK-UHFFFAOYSA-N',
  'status': 'SUCCESS',
  'notes': 'PubChem: IUPAC name resolved via InChIKey.'},
 {'compound_id': 'CMP-003',
  'input_smiles': 'CC(=O)Oc1ccccc1C(=O)O',
  'canonical_smiles': 'CC(=O)Oc1ccccc1C(=O)O',
  'iupac_name': '2-acetyloxybenzoic acid',
  'molecular_formula': 'C9H8O4',
  'exact_mass': 180.0423,
  'inchikey': 'BSYNRYMUTXBXSQ-UHFFFAOYSA-N',
  'status': 'SUCCESS',
  'notes': 'PubChem: IUPAC name resolved via InChIKey.'},
 {'compound_id': 'CMP-004',
  'input_smiles': 'O=C(O)c1ccccc